In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Block 0 diagnostic: does thermal inertia explain sols 1-170?

**A diagnostic, not a fit.** Nothing here changes the baseline or its bounds (`closed_ideas.md`: TI per block is closed for the baseline). Scales below the spec bound 0.7 are scanned only to see where block 0 points.

Question: the chosen residual model (`flann_safe`) carries a night TI effect of about 10 K, learned almost entirely from block 0 (TI 566-682 tiu, 9 K warm at night). If THEMIS TI is too high at block 0, one lower TI scale should fix block 0's nights, days and diurnal amplitude **together**. If night and day point to different scales, or sky IR explains the nights equally well, TI is not the (only) explanation.

Scans, with `g_diff` at the all-blocks fit:

1. `ti_scale` alone, `f_ir` at the all-blocks fit, for every training block: best scale by all-hours SSE, by night-only SSE (19-06 h), by day-only SSE (07-18 h), and the scale that matches the observed diurnal amplitude.
2. `ti_scale` x `f_ir` for block 0 and for all other blocks pooled: is the night bias a TI or a sky-IR signal?

Rows: main group only (no placeholder-tau sols, no sols 3763-3778).

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t

DRY_RUN = False
SUSPECT_SOLS = (3763, 3778)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
NIGHT_HOURS = [0, 1, 2, 3, 4, 5, 19, 20, 21, 22, 23]
TI_SCAN = np.round(np.linspace(0.3, 1.3, 21), 3) if not DRY_RUN else np.array([0.5, 0.9])
TI_SCAN_2D = np.round(np.linspace(0.3, 1.3, 11), 3) if not DRY_RUN else np.array([0.5, 0.9])
F_IR_SCAN = np.round(np.linspace(0.0, 0.06, 7), 3) if not DRY_RUN else np.array([0.02, 0.04])
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
OUT = paths.BLOCK0_OUT_DIR / ("dry_run" if DRY_RUN else "")
OUT.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = OUT / "block0_progress.log"
PROGRESS_LOG.write_text("")
fits = pd.read_csv(FIT_PARAMS).set_index("fold")
G_DIFF_ALL, F_IR_ALL, TI_SCALE_ALL = fits.loc["all training blocks", list(t.PARAM_NAMES)].to_numpy(dtype=float)
print(f"all-blocks fit: g_diff {G_DIFF_ALL:.4f}, f_ir {F_IR_ALL:.4f}, ti_scale {TI_SCALE_ALL:.4f}")
print("1D scan:", TI_SCAN.tolist())
print("2D scan:", TI_SCAN_2D.tolist(), "x", F_IR_SCAN.tolist())

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"
per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs = tbl.ground_temp_k.to_numpy()
block = tbl.block.to_numpy()
main = ~tbl.sol.between(*SUSPECT_SOLS).to_numpy() & ~tbl.tau_is_placeholder.to_numpy()
hour = (tbl.bin // 4).astype(int).to_numpy()
night = np.isin(hour, NIGHT_HOURS)
print(f"{int(main.sum())} main rows; block 0: {int((main & (block == 0)).sum())}")
print("THEMIS TI by block (main rows):")
print(tbl[main].groupby("block").ti_themis_tiu.agg(["min", "mean", "max"]).round(0).to_string())


def metrics(pred, mask):
    """RMSE, night/day bias, and diurnal amplitude error (hourly-mean max - min) on rows `mask`."""
    res = pred[mask] - obs[mask]
    hm = pd.DataFrame({"h": hour[mask], "p": pred[mask], "o": obs[mask]}).groupby("h").mean()
    return {"rmse_k": np.sqrt(np.mean(res ** 2)),
            "sse_all": float(res @ res),
            "sse_night": float((res[night[mask]] ** 2).sum()),
            "sse_day": float((res[~night[mask]] ** 2).sum()),
            "night_bias_k": res[night[mask]].mean(), "day_bias_k": res[~night[mask]].mean(),
            "amp_err_k": (hm.p.max() - hm.p.min()) - (hm.o.max() - hm.o.min())}


def log(line):
    print(line)
    with open(PROGRESS_LOG, "a") as fh:   # nbconvert shows output only at the end
        fh.write(line + "\n")

## Scan 1: `ti_scale` alone, per block

In [ ]:
rows1 = []
t0 = time.time()
for s in TI_SCAN:
    pred = evaluate((G_DIFF_ALL, F_IR_ALL, s))
    for b in blocks:
        m = main & (block == b)
        if m.any():
            rows1.append({"block": b, "ti_scale": s, **metrics(pred, m)})
    rows1.append({"block": "others", "ti_scale": s, **metrics(pred, main & (block != 0))})
    log(f"scan 1: ti_scale {s:.3f} | {time.time() - t0:.0f} s")
scan1 = pd.DataFrame(rows1)
scan1.to_csv(OUT / "scan_ti_scale_by_block.csv", index=False)


def best(g, col):
    return g.ti_scale.to_numpy()[np.argmin(g[col].to_numpy())]


def amp_zero(g):
    """ti_scale where the amplitude error crosses 0 (linear interpolation), NaN if it does not."""
    x, y = g.ti_scale.to_numpy(), g.amp_err_k.to_numpy()
    k = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]
    return float(x[k[0]] - y[k[0]] * (x[k[0] + 1] - x[k[0]]) / (y[k[0] + 1] - y[k[0]])) if len(k) else np.nan


summary1 = []
for b, g in scan1.groupby("block", sort=False):
    g = g.sort_values("ti_scale")
    s_all = best(g, "sse_all")
    at = g[g.ti_scale == s_all].iloc[0]
    summary1.append({"block": b, "best_all": s_all, "best_night": best(g, "sse_night"), "best_day": best(g, "sse_day"),
                     "amp_match": amp_zero(g), "rmse_at_best_k": at.rmse_k,
                     "night_bias_at_best_k": at.night_bias_k, "day_bias_at_best_k": at.day_bias_k})
summary1 = pd.DataFrame(summary1).set_index("block")
ti_mean = tbl[main].groupby("block").ti_themis_tiu.mean()
summary1["implied_ti_tiu"] = [summary1.best_all[b] * (ti_mean[b] if b != "others" else ti_mean.drop(index=0).mean())
                              for b in summary1.index]
summary1.to_csv(OUT / "best_ti_scale_by_block.csv")
pd.options.display.float_format = "{:.2f}".format
print("Best ti_scale per block (g_diff, f_ir at the all-blocks fit; scan step 0.05):")
print(summary1.to_string())

## Scan 2: `ti_scale` x `f_ir`, block 0 vs the other blocks

In [ ]:
rows2 = []
t0 = time.time()
for f in F_IR_SCAN:
    for s in TI_SCAN_2D:
        pred = evaluate((G_DIFF_ALL, f, s))
        for name, m in (("block 0", main & (block == 0)), ("others", main & (block != 0))):
            rows2.append({"rows": name, "f_ir": f, "ti_scale": s, **metrics(pred, m)})
        log(f"scan 2: f_ir {f:.3f}, ti_scale {s:.3f} | {time.time() - t0:.0f} s")
scan2 = pd.DataFrame(rows2)
scan2.to_csv(OUT / "scan_ti_scale_f_ir.csv", index=False)
for name, g in scan2.groupby("rows"):
    b = g.loc[g.sse_all.idxmin()]
    print(f"{name}: best f_ir {b.f_ir:.3f}, ti_scale {b.ti_scale:.2f} | RMSE {b.rmse_k:.2f} K, "
          f"night bias {b.night_bias_k:+.2f} K, day bias {b.day_bias_k:+.2f} K, amplitude error {b.amp_err_k:+.2f} K")
    for col in ["rmse_k", "night_bias_k", "day_bias_k", "amp_err_k"]:
        print(f"  {name} {col} (rows f_ir, columns ti_scale):")
        print(g.pivot(index="f_ir", columns="ti_scale", values=col).to_string())

## Block 0 diurnal shape

In [ ]:
b0 = main & (block == 0)
best0 = summary1.loc[0, "best_all"]
g2 = scan2[scan2.rows == "block 0"]
best2 = g2.loc[g2.sse_all.idxmin()]
cases = {"all-blocks fit": (G_DIFF_ALL, F_IR_ALL, TI_SCALE_ALL),
         f"ti_scale {best0:.2f} (scan 1 best)": (G_DIFF_ALL, F_IR_ALL, best0),
         f"f_ir {best2.f_ir:.3f}, ti_scale {best2.ti_scale:.2f} (scan 2 best)": (G_DIFF_ALL, best2.f_ir, best2.ti_scale)}
prof = pd.DataFrame({"REMS": pd.Series(obs[b0]).groupby(hour[b0]).mean()})
for name, p in cases.items():
    prof[name] = pd.Series(evaluate(p)[b0]).groupby(hour[b0]).mean()
prof.to_csv(OUT / "block0_hourly_profile.csv")
print((prof.drop(columns="REMS").sub(prof.REMS, axis=0)).rename(columns=lambda c: c + " bias").to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
ax.plot(prof.index, prof.REMS, "k.-", label="REMS")
for (name, _), st in zip(cases.items(), ["r-", "g-", "b--"]):
    ax.plot(prof.index, prof[name], st, label=name)
ax.set(xlabel="LTST hour", ylabel="ground temperature (K)", title="Block 0 (sols 1-170), hourly mean")
ax.legend(fontsize=7)
ax = axes[1]
for b, g in scan1[scan1.block.isin([0, "others"])].groupby("block"):
    ax.plot(g.ti_scale, g.night_bias_k, "-", label=f"{'block 0' if b == 0 else 'other blocks'} night")
    ax.plot(g.ti_scale, g.day_bias_k, "--", label=f"{'block 0' if b == 0 else 'other blocks'} day")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(TI_SCALE_ALL, color="grey", ls=":", label="all-blocks fit")
ax.set(xlabel="ti_scale", ylabel="bias (K)", title="Scan 1: bias vs ti_scale")
ax.legend(fontsize=7)
ax = axes[2]
ax.plot(summary1.drop(index="others").index.astype(str), summary1.drop(index="others").best_night, "o", label="best by night")
ax.plot(summary1.drop(index="others").index.astype(str), summary1.drop(index="others").best_day, "s", label="best by day")
ax.plot(summary1.drop(index="others").index.astype(str), summary1.drop(index="others").amp_match, "^", label="amplitude match")
ax.axhline(TI_SCALE_ALL, color="grey", ls=":")
ax.set(xlabel="block", ylabel="ti_scale", title="Best ti_scale per block (scan 1)")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "block0_diagnostic.png", dpi=120)
print("saved to", OUT)